In [1]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats
import statsmodels.api as sm
import warnings
warnings.filterwarnings('ignore')

from src.config import PROCESSED_DIR, REPORTS_DIR

df = pd.read_parquet(PROCESSED_DIR / 'jakarta_master.parquet')
df['date'] = pd.to_datetime(df['date'])
df = df.set_index('date')

# Focus 2019-2022 (sufficient coverage)
df = df['2019':'2022'].copy()
df_valid = df.dropna(subset=['pm25_mean']).copy()

print(f'Loaded: {df_valid.shape}')
print(f'Valid records: {len(df_valid)}')

Loaded: (1182, 26)
Valid records: 1182


In [2]:
weather_vars = ['temp_mean', 'precipitation', 'windspeed_max',
                'humidity_mean', 'temp_max', 'temp_min']

corr_results = []
for var in weather_vars:
    rho, pval = stats.spearmanr(
        df_valid['pm25_mean'],
        df_valid[var],
        nan_policy='omit'
    )
    corr_results.append({
        'Variable'    : var,
        'Spearman_rho': round(rho, 4),
        'p_value'     : round(pval, 4),
        'Significant' : 'YES' if pval < 0.05 else 'NO',
        'Strength'    : 'Strong' if abs(rho) > 0.5 else 'Moderate' if abs(rho) > 0.3 else 'Weak',
    })

corr_df = pd.DataFrame(corr_results).sort_values('Spearman_rho')
print('=== Spearman Correlation: PM2.5 vs Weather Variables ===')
print(corr_df.to_string(index=False))

=== Spearman Correlation: PM2.5 vs Weather Variables ===
     Variable  Spearman_rho  p_value Significant Strength
precipitation       -0.4711   0.0000         YES Moderate
humidity_mean       -0.4436   0.0000         YES Moderate
windspeed_max       -0.2641   0.0000         YES     Weak
     temp_min       -0.0152   0.6017          NO     Weak
    temp_mean        0.1716   0.0000         YES     Weak
     temp_max        0.2873   0.0000         YES     Weak


In [3]:
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        'PM2.5 vs Wind Speed',
        'PM2.5 vs Precipitation',
        'PM2.5 vs Humidity',
        'PM2.5 vs Temperature',
    ]
)

plot_vars = [
    ('windspeed_max', 'Wind Speed Max (km/h)', 1, 1),
    ('precipitation', 'Precipitation (mm)',    1, 2),
    ('humidity_mean', 'Humidity Mean (%)',      2, 1),
    ('temp_mean',     'Temperature Mean (C)',   2, 2),
]

for var, label, row, col in plot_vars:
    rho_val = corr_df[corr_df['Variable'] == var]['Spearman_rho'].values[0]

    fig.add_trace(go.Scatter(
        x=df_valid[var],
        y=df_valid['pm25_mean'],
        mode='markers',
        marker=dict(
            size=4,
            color=df_valid['pm25_mean'],
            colorscale='RdYlGn_r',
            opacity=0.5,
        ),
        name=f'rho={rho_val:.3f}',
        showlegend=False,
    ), row=row, col=col)

    fig.update_xaxes(title_text=label, row=row, col=col)
    fig.update_yaxes(title_text='PM2.5 (ug/m3)', row=row, col=col)

fig.update_layout(
    title='Jakarta PM2.5 vs Weather Variables (Spearman)',
    height=700,
    template='plotly_white',
)

fig.show()
out = str(REPORTS_DIR / 'figures' / '05_weather_scatter.html')
fig.write_html(out)
print(f'Saved: {out}')

Saved: C:\AQSEL\me\KARIR\Portofolio Setelah Wisuda\EDA Project Indonesia Urban Air Quality Analysis\reports\figures\05_weather_scatter.html


In [4]:
lag_corrs = []
for lag in range(0, 8):
    shifted = df_valid['precipitation'].shift(lag)
    mask = shifted.notna() & df_valid['pm25_mean'].notna()
    rho, pval = stats.spearmanr(
        df_valid['pm25_mean'][mask],
        shifted[mask]
    )
    lag_corrs.append({
        'lag_days'    : lag,
        'spearman_rho': round(rho, 4),
        'p_value'     : round(pval, 4),
    })

lag_df = pd.DataFrame(lag_corrs)
print('=== Precipitation Lag Analysis ===')
print(lag_df.to_string(index=False))

optimal_lag = lag_df.loc[lag_df['spearman_rho'].abs().idxmax(), 'lag_days']
print(f'\nOptimal lag: {optimal_lag} days')
print(f'Rain cleans air most effectively after {optimal_lag} day(s)')

=== Precipitation Lag Analysis ===
 lag_days  spearman_rho  p_value
        0       -0.4711      0.0
        1       -0.4853      0.0
        2       -0.4143      0.0
        3       -0.3979      0.0
        4       -0.3567      0.0
        5       -0.3640      0.0
        6       -0.3745      0.0
        7       -0.3630      0.0

Optimal lag: 1 days
Rain cleans air most effectively after 1 day(s)


In [5]:
season_stats = df_valid.groupby('season')['pm25_mean'].agg([
    'count', 'mean', 'median', 'std',
    lambda x: x.quantile(0.25),
    lambda x: x.quantile(0.75),
]).round(2)
season_stats.columns = ['count', 'mean', 'median', 'std', 'q25', 'q75']

print('=== PM2.5 by Season ===')
print(season_stats)

dry = df_valid[df_valid['season'] == 'dry']['pm25_mean'].dropna()
wet = df_valid[df_valid['season'] == 'wet']['pm25_mean'].dropna()
stat, pval = stats.mannwhitneyu(dry, wet, alternative='greater')

print(f'\nMann-Whitney U test (dry > wet):')
print(f'  U-statistic: {stat:.0f}')
print(f'  p-value: {pval:.6f}')
print(f'  Result: {"Dry significantly higher" if pval < 0.05 else "Not significant"}')
print(f'\n  Dry season mean : {dry.mean():.1f} ug/m3')
print(f'  Wet season mean : {wet.mean():.1f} ug/m3')
print(f'  Difference      : {dry.mean() - wet.mean():.1f} ug/m3 ({(dry.mean()-wet.mean())/wet.mean()*100:.1f}% higher)')

=== PM2.5 by Season ===
        count   mean  median    std    q25    q75
season                                           
dry       595  44.48   40.42  36.56  30.76  49.60
wet       587  31.42   26.71  33.27  17.44  37.58

Mann-Whitney U test (dry > wet):
  U-statistic: 255884
  p-value: 0.000000
  Result: Dry significantly higher

  Dry season mean : 44.5 ug/m3
  Wet season mean : 31.4 ug/m3
  Difference      : 13.1 ug/m3 (41.5% higher)


In [6]:
df_valid['is_weekend'] = df_valid['dayofweek'].isin([5, 6])

weekend_stats = df_valid.groupby('is_weekend')['pm25_mean'].agg(
    ['count', 'mean', 'median', 'std']
).round(2)
weekend_stats.index = ['Weekday', 'Weekend']

print('=== Weekend vs Weekday PM2.5 ===')
print(weekend_stats)

weekday = df_valid[df_valid['is_weekend'] == False]['pm25_mean'].dropna()
weekend = df_valid[df_valid['is_weekend'] == True]['pm25_mean'].dropna()
stat, pval = stats.mannwhitneyu(weekday, weekend, alternative='greater')

print(f'\nMann-Whitney U test (weekday > weekend):')
print(f'  p-value: {pval:.4f}')
print(f'  Result: {"Weekday significantly higher" if pval < 0.05 else "Not significant"}')
print(f'\n  Weekday mean : {weekday.mean():.1f} ug/m3')
print(f'  Weekend mean : {weekend.mean():.1f} ug/m3')
diff = weekday.mean() - weekend.mean()
print(f'  Difference   : {diff:.1f} ug/m3 ({"traffic signature detected" if pval < 0.05 else "no traffic signature"})')

=== Weekend vs Weekday PM2.5 ===
         count   mean  median    std
Weekday    849  39.12   33.83  39.71
Weekend    333  35.14   33.96  21.36

Mann-Whitney U test (weekday > weekend):
  p-value: 0.3771
  Result: Not significant

  Weekday mean : 39.1 ug/m3
  Weekend mean : 35.1 ug/m3
  Difference   : 4.0 ug/m3 (no traffic signature)


In [7]:
import json

summary = {
    'spearman_correlations': corr_df.to_dict(),
    'lag_analysis'         : lag_df.to_dict(),
    'season_stats'         : season_stats.to_dict(),
    'weekend_effect_pval'  : pval,
    'weekday_mean'         : weekday.mean(),
    'weekend_mean'         : weekend.mean(),
}

out_json = REPORTS_DIR / 'weather_correlation_summary.json'
with open(out_json, 'w') as f:
    json.dump(summary, f, indent=2, default=str)

print(f'Summary saved: {out_json}')
print('\n=== ANALYSIS COMPLETE ===')
print('Notebooks done:')
print('  05_eda_temporal.ipynb')
print('  06_covid_analysis.ipynb')
print('  07_weather_correlation.ipynb')
print('\nNext: Streamlit Dashboard')

Summary saved: C:\AQSEL\me\KARIR\Portofolio Setelah Wisuda\EDA Project Indonesia Urban Air Quality Analysis\reports\weather_correlation_summary.json

=== ANALYSIS COMPLETE ===
Notebooks done:
  05_eda_temporal.ipynb
  06_covid_analysis.ipynb
  07_weather_correlation.ipynb

Next: Streamlit Dashboard
